In [1]:
import jax
import jax.numpy as jnp 
import jax.random as jrandom

import matplotlib.pyplot as plt

from IPython.display import display

jax.config.update("jax_enable_x64", True)
display(jax.default_backend())
display(jnp.array(1.0).dtype)

'gpu'

dtype('float64')

In [4]:
# Static Config Parameters
d = 16
niterations = 30
nsamples = 3_000
nfim_iterations = 500

key = jrandom.key(d)

loc_shift = jnp.array(125.0)
scale_shift = jnp.array(3.0)

loc_naught_shift = jnp.array(25.0)
scale_naught_shift = jnp.array(1.75)

In [5]:
# Maximum Likelihood Estimator (Analytical) for Gaussian (Univariate)

key, loc_true_key, scale_true_key = jrandom.split(key, 3)
loc_true = jrandom.normal(loc_true_key, shape=(d, )) * loc_shift
scale_true = jnp.abs(jrandom.normal(scale_true_key, shape=(d, ))) * scale_shift

# Generate all the iteration keys 
iteration_keys = jrandom.split(key, niterations)

def one_iteration_mle(iteration_key: jax.Array):
    def one_sample(sample_key: jax.Array):
        # Sample from the True Distribution
        sample = jrandom.normal(sample_key, shape=(d, 1)) * scale_true + loc_true
        return sample
    
    # Generate all the samples keys 
    sample_keys = jrandom.split(iteration_key, nsamples)
    # Sample nsamples
    samples = jax.vmap(one_sample)(sample_keys)
    # Calculate Mean & Variance for Maximum Likelihood Estimator
    loc_mle = jnp.mean(samples, axis=0)
    scale_mle = jnp.var(samples, axis=0)

    return loc_mle, scale_mle

loc_estimates_mle, scale_estimates_mle = jax.vmap(one_iteration_mle)(iteration_keys)

loc_bias_mle = jnp.linalg.norm(jnp.mean(loc_estimates_mle, axis=0) - loc_true)
loc_variance_mle = jnp.linalg.norm(jnp.var(loc_estimates_mle, axis=0))

scale_bias_mle = jnp.linalg.norm(jnp.mean(scale_estimates_mle, axis=0) - jnp.pow(scale_true, 2))
scale_variance_mle = jnp.linalg.norm(jnp.var(scale_estimates_mle, axis=0))

display("Mean Estimate Bias & Variance (ML Estimator)")
display(f"L2Norm(Bias) = {loc_bias_mle}")
display(f"L2Norm(Variance) = {loc_variance_mle}")

display("Variance Estimate Bias & Variance (ML Estimator)")
display(f"L2Norm(Bias) = {scale_bias_mle}")
display(f"L2Norm(Variance) = {scale_variance_mle}")

'Mean Estimate Bias & Variance (ML Estimator)'

'L2Norm(Bias) = 0.1440328946289325'

'L2Norm(Variance) = 0.11973318570532027'

'Variance Estimate Bias & Variance (ML Estimator)'

'L2Norm(Bias) = 1.8235730122236382'

'L2Norm(Variance) = 12.82387712363654'

In [6]:
# Cramer-Rao Lower Bound Calculation for ML Estimator

def log_likelihood_univariate(params, x):
    loc, var = params[0], params[1]
    return jax.scipy.stats.norm.logpdf(x, loc, var)

log_likelihood_univariate_hessian = jax.hessian(log_likelihood_univariate)

# Fischer Information Matrix is -E[Hessian[Likelihood]]
# We'll average Fischer Information Matrix

def one_fim_iteration(loc_t, var_t, fim_key):
    samples = jrandom.normal(fim_key, shape=(nsamples,)) * jnp.sqrt(var_t) + loc_t
    params = jnp.array([loc_t, var_t])
    hessians = jax.vmap(log_likelihood_univariate_hessian, in_axes=(None, 0))(params, samples)
    fim_i = -jnp.mean(hessians, axis=0)
    return fim_i

fim_keys = jrandom.split(key, d)
fims = jax.vmap(one_fim_iteration)(loc_true, jnp.square(scale_true), fim_keys)
crlb = jnp.linalg.inv(fims * nsamples)

display("Cramer Rao Lower Bound")
display(f"Mean for CRLB = {crlb[:, 0, 0]}")
display(f"Variance for CRLB = {crlb[:, 1, 1]}")

'Cramer Rao Lower Bound'

'Mean for CRLB = [2.07815430e-05 8.76636379e-03 3.46080416e-06 1.93338593e-03\n 1.67241101e-02 1.00872618e-02 6.61670696e-02 4.10354273e-01\n 5.20243641e-03 4.71035853e-04 8.75222461e-01 3.41694271e-02\n 3.12035136e-04 1.00624422e-01 1.27841276e+00 1.84009284e-01]'

'Variance for CRLB = [ 1.95719691e-06 -2.18810803e-02  1.22493101e-07  6.95611581e-03\n -2.94223617e-02 -2.22985618e-02 -8.28649771e-02 -4.48193446e-01\n -1.95729393e-02  2.84906113e-04 -9.27325418e-01 -4.89659760e-02\n  1.40824997e-04 -1.21959467e-01 -1.34406343e+00 -2.08773944e-01]'

In [ ]:
# Maximum Aposteriori Estimator (Analytical) for Gaussian (Univariate)

key, loc_true_key, scale_true_key, loc_naught_key, scale_naught_key = jrandom.split(key, 5)
loc_true = jrandom.normal(loc_true_key, shape=(d, 1)) * loc_shift
scale_true = jrandom.normal(scale_true_key, shape=(d, 1)) * scale_shift

loc_naught = jrandom.normal(loc_naught_key, shape=(d, 1)) * loc_shift
scale_naught = jrandom.normal(scale_naught_key, shape=(d, 1)) * scale_shift

scale_naught_inv_square = jnp.square(1/scale_naught)

# Generate all the iteration keys 
iteration_keys = jrandom.split(key, niterations)

def one_iteration_map(iteration_key: jax.Array):
    def one_sample(sample_key: jax.Array):
        # Sample from the True Distribution
        sample = jrandom.normal(sample_key, shape=(d, 1)) * scale_true + loc_true
        return sample
    
    # Generate all the samples keys 
    sample_keys = jrandom.split(iteration_key, nsamples)
    # Sample nsamples
    samples = jax.vmap(one_sample)(sample_keys)
    # Calculate Mean & Variance for Maximum Aposteriori Estimator
    scale_inv_square = jnp.square(1/(jnp.var(samples, axis=0)))

    scale_map = 1/(nsamples*scale_inv_square + scale_naught_inv_square)
    loc_map = scale_map * (jnp.sum(samples, axis=0) * scale_inv_square + loc_naught*scale_naught_inv_square)
    return loc_map, scale_map

loc_estimates_map, scale_estimates_map = jax.vmap(one_iteration_map)(iteration_keys)

loc_bias_map = jnp.linalg.norm(jnp.mean(loc_estimates_map, axis=0) - loc_true)
loc_variance_map = jnp.linalg.norm(jnp.var(loc_estimates_map, axis=0))

display("Mean Estimate Bias & Variance (MAP Estimation)")
display(f"L2Norm(Bias) = {loc_bias_map}")
display(f"L2Norm(Variance) = {loc_variance_map}")

'Mean Estimate Bias & Variance (MAP Estimation)'

'L2Norm(Bias) = 75.18457627440588'

'L2Norm(Variance) = 10.09031544795999'